In [18]:
import psycopg
from psycopg.types.json import Jsonb
from psycopg.rows import dict_row

In [6]:
conn = psycopg.connect(
    host="localhost",
    port=5432,
    dbname="initdb", 
    user="admin",
    password="qwer123456",
)


print("连接成功:", conn.info.server_version) 

连接成功: 170010


In [7]:
with conn.cursor() as cur:
    cur.execute("""
        CREATE TABLE IF NOT EXISTS students (
            id      SERIAL PRIMARY KEY, 
            name    VARCHAR(50) NOT NULL,
            profile JSONB     
        )
    """)
conn.commit()
print("建表完成")

建表完成


In [9]:
with conn.cursor() as cur:
    cur.execute(
        "INSERT INTO students (name, profile) VALUES (%s, %s)",
        ("Alice", Jsonb({"age": 20, "score": 88.5})),
    )

conn.commit() 
print("插入数据完成")

插入数据完成


In [10]:
rows = [
    ("Carol", Jsonb({"age": 19, "score": 76.0})),
    ("Dave", Jsonb({"age": 23, "score": 82.5})),
    ("Eve", Jsonb({"age": 21, "score": 95.0})),
]
with conn.cursor() as cur:
    cur.executemany(
        "INSERT INTO students (name, profile) VALUES (%s, %s)",
        rows,
    )
conn.commit()
print(f"批量插入{len(rows)}条")

批量插入3条


In [ ]:
with conn.cursor() as cur:
    cur.execute("SELECT id, name, profile FROM students ORDER BY id")
    rows = cur.fetchall()

    columns = [d.name for d in cur.description]
    print("列名:", columns)
    for row in rows:
        print(row)  
        print(type(row))  # tuple
        print(type(row[2]))  # dict(JSONB)
        print()

列名: ['id', 'name', 'profile']
(1, 'Alice', {'age': 20, 'score': 88.5})
<class 'tuple'>
<class 'dict'>

(2, 'Carol', {'age': 19, 'score': 76.0})
<class 'tuple'>
<class 'dict'>

(3, 'Dave', {'age': 23, 'score': 82.5})
<class 'tuple'>
<class 'dict'>

(4, 'Eve', {'age': 21, 'score': 95.0})
<class 'tuple'>
<class 'dict'>



In [ ]:
with conn.cursor(row_factory=dict_row) as cur:
    cur.execute("SELECT id, name, profile FROM students ORDER BY id")
    for row in cur:
        print(row)

{'id': 1, 'name': 'Alice', 'profile': {'age': 20, 'score': 88.5}}
Alice -> 88.5
{'id': 2, 'name': 'Carol', 'profile': {'age': 19, 'score': 76.0}}
Carol -> 76.0
{'id': 3, 'name': 'Dave', 'profile': {'age': 23, 'score': 82.5}}
Dave -> 82.5
{'id': 4, 'name': 'Eve', 'profile': {'age': 21, 'score': 95.0}}
Eve -> 95.0


In [21]:
with conn.cursor() as cur:
    cur.execute(
        "UPDATE students SET profile = jsonb_set(profile, '{score}', %s) WHERE name = %s",
        (Jsonb(90.0), "Alice"),
    )
    print("更新行数:", cur.rowcount)
    
    cur.execute("DELETE FROM students WHERE name = %s", ("Eve",))
    print("删除行数:", cur.rowcount)

conn.commit()

更新行数: 1
删除行数: 1
